In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain.tools import tool
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [11]:
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # start_date=None,
    # end_date=None,
    # include_domains=None,
    # exclude_domains=None,
    # include_usage= False
)

In [15]:
model = init_chat_model("openrouter:google/gemini-3.8-flash")

In [ ]:
agent = create_agent(
    model=model,
    system_prompt="You are a helpful assistant. Use web search to find accurate, up-to-date information.",
    tools=[tavily_search_tool]
)

In [19]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the most popular sport in the world? Include only Wikipedia sources. "}]}
)
print(result['messages'][-1].content)

The most popular sport in the world is **association football** (commonly known as **football** or **soccer**). 

According to Wikipedia, with an estimated 250 million players active across more than 200 countries and territories, association football is recognized as the world's most popular sport in terms of both participation and spectatorship.

### Wikipedia Sources:
* [Association football - Wikipedia](https://en.wikipedia.org/wiki/Association_football)
* [Football - Wikipedia](https://en.wikipedia.org/wiki/Football)


In [ ]:
agent.stream()

In [ ]:
agent.batch()

In [22]:
@tool
def siparis_durumu(siparis_no: str) -> str:
    """Müşteri siparişinin güncel durumunu getirir."""
    # Gerçek uygulamada burada sipariş sistemi sorgulanır
    return f"Sipariş {siparis_no} kargoda, 2 gün içinde teslim edilecek."


agent = create_agent(model=model, tools=[siparis_durumu])

sonuc = agent.invoke({"messages": [{"role": "user", "content": "12345 nolu siparişim nerede?"}]})
sonuc

{'messages': [HumanMessage(content='12345 nolu siparişim nerede?', additional_kwargs={}, response_metadata={}, id='77d6fa50-3d0d-4155-8caf-f28e52ab6539'),
  AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'AY89a18I/vkVSLO11qJgvsxuPAmSAKokpLGxAi3jAasJq2vtVJUrz0FdKoOUyojC31LMxghuxPLpeZXY+OkjirhA46vbGmJUwufwar6J0zoUGCWhgNriIG9xi6D7Zubt+PNA6VqHEaKSO3+BLdwsoEFeKQCFoNWWLdAii4OmeVgiaMJCoMZrbTvRM9U3gPNUTC38VHec7uKvMoN58nKuijkJjQheZMFSEq/wG7c4e3S1oesFoqgXuqk3aBo3iWB4zV2rLNLu6lVLJPTLV6lR6xRRARd0IKhR8PVGZ92JMjY2MC4w/Jm9ptRhw3O0nksLCuXxEUd0wWhI1ye6t5wiDg8GuHTZN7wBBt5QI8MSpgG48+3mqCuiNlc01VhXiYEPsalGU4F71gBEni/BAe86V8r8', 'type': 'reasoning.encrypted', 'format': 'google-gemini-v1', 'id': 'call_476832', 'index': 0}]}, response_metadata={'model_name': 'google/gemini-3.8-flash', 'id': 'gen-1791914875-jprlOEYsygyAV1xto6gU', 'created': 1791914875, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000357, 'cost_

In [21]:
sonuc['messages'][-1].content

'Sipariş 12345 kargoda, 2 gün içinde teslim edilecek.'

In [35]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage

In [26]:
@dataclass
class UserContext:
    user_id: str
    user_age: int

In [25]:
KULLANICILAR = {
    "user123": {"ad": "Ayşe Yılmaz", "bakiye": 5000},
    "user456": {"ad": "Mehmet Kaya", "bakiye": 1200},
}

In [27]:
@tool
def hesap_bilgisi(runtime: ToolRuntime[UserContext]) -> str:
    """Mevcut kullanıcının hesap bilgisini getirir."""
    kullanici = KULLANICILAR.get(runtime.context.user_id)
    if kullanici is None:
        return "Kullanıcı bulunamadı"
    return f"{kullanici['ad']}, bakiye: {kullanici['bakiye']} TL"

In [37]:
@tool
def get_last_tool_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the Tool."""
    messages = runtime.state["messages"]

    # Find the last tool message
    for message in reversed(messages):
        if isinstance(message, ToolMessage):
            return message.content

    return "No tool messages found"

In [ ]:
tools_for_complaint_request = []
tools_for_customer_response = []

In [38]:
agent = create_agent(
    model=model,
    tools=[hesap_bilgisi],
    context_schema=UserContext,
    system_prompt="Sen bir bankacılık asistanısın.",
)

In [39]:
sonuc = agent.invoke(
    {"messages": [{"role": "user", "content": "Bakiyem ne kadar? ve en son tool mesajını bana yorumla."}]},
    context=UserContext(user_id="user123", user_age=34),
)

In [40]:
sonuc

{'messages': [HumanMessage(content='Bakiyem ne kadar? ve en son tool mesajını bana yorumla.', additional_kwargs={}, response_metadata={}, id='99f9a129-14c2-4fd1-8f52-cd792f87a4bd'),
  AIMessage(content='', additional_kwargs={'reasoning_details': [{'data': 'AY89a18E0s88xzqhB7DfR4nhqRNjbR3GvrUnknkA4D2Iq/bE2C0JL/qZhCd69KRvfkN5rru4CwJj3xk8CmQRc+jLBG1vq2TTYXTjJMJmUkaHJQ0eCgBxnwUpy0/GcQ18Y4ii2olrD4HwDyL0+H+kRRGWr27NOo9rC+bXgZSSbc6QSwlMl1TIbsoE8hqwNkcX68Ies1vJd+7imZWn+clw9oWE5lkJIGfWbmHAnc8GSUcqiP5tyHKRVdgPqFvENBL/9FUyewC9Ni8vh/SmP3kK+3ttuJb9IoQB6+QFR5sY5encEatxHHB4iEf5SH4FgQA6t/HgC0sm5d+ZZ5qHCdtxjrATwb43TOIHRTkMZDUg8Q9bO+jme8fgZAdD7ZEvDA9dRb47u9M0XQgI+D3aTHZhi9gCNYY+FYF6qJ5ga+R7XMcFBQOZarza4ksg4HSKJU4JN9KwgLJr6Rh7CzLW16lZZnd29cX2nKAFkMs8PBXPqwdqBk0mIKsrXKpcFWPrnSlOp7fI3yO8v7tsxpOxI60DgVttWk1oWH8aymw/fnIAA8QP13NuwjHNjXJdlrUVEOirNiQ+S9VRJEPF6ElYXOEoj2eO', 'type': 'reasoning.encrypted', 'format': 'google-gemini-v1', 'id': 'call_1217076', 'index': 0}]}, response_metadata={'model_name': 'google/g